# Task 3.7

This notebook answers Task 3.7. The goal is to maximize the minimum of the three mean qualities while producing 5000 litres of amber beer.

In [8]:
from brew_utils import *
from scipy.optimize import linprog
import numpy as np
import pandas as pd

## 3.7 Problem Setup

In Task 3.7, the objective is no longer cost minimization. Instead, we maximize the minimum of:
- the average malt-source quality
- the average hop quality
- the average yeast quality

The cost is unlimited, but the beer must still satisfy the volume and amber-color constraints.

In [9]:
barley_cost = barley["cost"].to_numpy(dtype=float)
barley_ebc = barley["EBC"].to_numpy(dtype=float)
barley_q = barley["quality"].to_numpy(dtype=float)

malt_cost = malts["cost"].to_numpy(dtype=float)
malt_ebc = malts["EBC"].to_numpy(dtype=float)
malt_q = malts["quality"].to_numpy(dtype=float)

extract_cost = maltextracts["cost"].to_numpy(dtype=float)
extract_ebc = maltextracts["EBC"].to_numpy(dtype=float)
extract_q = maltextracts["quality"].to_numpy(dtype=float)

hop_q = hops["quality"].to_numpy(dtype=float)
yeast_q = yeasts["quality"].to_numpy(dtype=float)

nB = len(barley)
nM = len(malts)
nE = len(maltextracts)
nH = len(hops)
nY = len(yeasts)

iB0 = 0
iM0 = iB0 + nB
iE0 = iM0 + nM
iH0 = iE0 + nE
iY0 = iH0 + nH
iQ = iY0 + nY
iZm = iQ + 1
iZs = iZm + 1
nvars = iZs + 1

def slice_B():
    return slice(iB0, iB0 + nB)

def slice_M():
    return slice(iM0, iM0 + nM)

def slice_E():
    return slice(iE0, iE0 + nE)

def slice_H():
    return slice(iH0, iH0 + nH)

def slice_Y():
    return slice(iY0, iY0 + nY)

## 3.7 Linear Model

To maximize the minimum quality, we introduce a new variable `q_min`. This variable represents the lowest of the three average qualities.

The model then maximizes `q_min` subject to:
- the volume requirement
- the amber EBC interval
- the hop and yeast amount constraints
- process logic constraints
- each of the three average qualities being at least `q_min`

This keeps the problem linear.

In [10]:
def build_quality_model(V_beer, EBC_min, EBC_max, z_fix=None):
    m_tot = malt_extract_eq_needed_for_beer(V_beer)
    H_tot = hops_needed_for_beer(V_beer)
    Y_tot = yeast_needed_for_beer(V_beer)
    gamma = (SG - 1.0) / 0.0344

    M_B = m_tot / BARLEY_TO_MALTEXTRACT_EQ
    M_M = m_tot / MALT_TO_MALTEXTRACT_EQ

    c = np.zeros(nvars)
    c[iQ] = -1.0

    A_eq = []
    b_eq = []

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALTEXTRACT_EQ
    row[slice_M()] = MALT_TO_MALTEXTRACT_EQ
    row[slice_E()] = 1.0
    A_eq.append(row)
    b_eq.append(m_tot)

    row = np.zeros(nvars)
    row[slice_H()] = 1.0
    A_eq.append(row)
    b_eq.append(H_tot)

    row = np.zeros(nvars)
    row[slice_Y()] = 1.0
    A_eq.append(row)
    b_eq.append(Y_tot)

    A_ub = []
    b_ub = []

    if np.isfinite(EBC_max):
        row = np.zeros(nvars)
        row[slice_B()] = gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
        row[slice_M()] = gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
        row[slice_E()] = gamma * extract_ebc
        A_ub.append(row)
        b_ub.append(EBC_max * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = -gamma * BARLEY_TO_MALTEXTRACT_EQ * barley_ebc
    row[slice_M()] = -gamma * MALT_TO_MALTEXTRACT_EQ * malt_ebc
    row[slice_E()] = -gamma * extract_ebc
    A_ub.append(row)
    b_ub.append(-EBC_min * m_tot)

    row = np.zeros(nvars)
    row[slice_B()] = -BARLEY_TO_MALTEXTRACT_EQ * barley_q
    row[slice_M()] = -MALT_TO_MALTEXTRACT_EQ * malt_q
    row[slice_E()] = -extract_q
    row[iQ] = m_tot
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[slice_H()] = -hop_q
    row[iQ] = H_tot
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[slice_Y()] = -yeast_q
    row[iQ] = Y_tot
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[slice_B()] = 1.0
    row[iZm] = -M_B
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[slice_B()] = BARLEY_TO_MALT
    row[slice_M()] = 1.0
    row[iZs] = -M_M
    A_ub.append(row)
    b_ub.append(0.0)

    row = np.zeros(nvars)
    row[iZm] = 1.0
    row[iZs] = -1.0
    A_ub.append(row)
    b_ub.append(0.0)

    bounds = [(0, None)] * nvars
    bounds[iQ] = (0, 5)

    if z_fix is None:
        bounds[iZm] = (0, 1)
        bounds[iZs] = (0, 1)
    else:
        bounds[iZm] = (z_fix[0], z_fix[0])
        bounds[iZs] = (z_fix[1], z_fix[1])

    return {
        "c": c,
        "A_eq": np.array(A_eq, dtype=float),
        "b_eq": np.array(b_eq, dtype=float),
        "A_ub": np.array(A_ub, dtype=float),
        "b_ub": np.array(b_ub, dtype=float),
        "bounds": bounds,
        "m_tot": m_tot,
        "H_tot": H_tot,
        "Y_tot": Y_tot,
    }

## 3.7 Solve the MILP by Brute Force

As in the earlier tasks, we solve the original MILP by brute force over the valid process choices:
- `(0, 0)` only malt extract
- `(0, 1)` mashing only
- `(1, 1)` malting and mashing

For each case, the process variables are fixed and the remaining linear program is solved.

In [11]:
def solve_with_fixed_processes(V_beer, EBC_min, EBC_max, z_malt, z_mash):
    model = build_quality_model(V_beer, EBC_min, EBC_max, z_fix=(z_malt, z_mash))
    res = linprog(
        c=model["c"],
        A_ub=model["A_ub"],
        b_ub=model["b_ub"],
        A_eq=model["A_eq"],
        b_eq=model["b_eq"],
        bounds=model["bounds"],
        method="highs",
    )
    return model, res

def solve_bruteforce(V_beer, EBC_min, EBC_max):
    candidates = [(0, 0), (0, 1), (1, 1)]
    best = None
    all_results = []

    for z_fix in candidates:
        model, res = solve_with_fixed_processes(V_beer, EBC_min, EBC_max, *z_fix)
        info = {
            "z_fix": z_fix,
            "success": res.success,
            "q_min": -res.fun if res.success else np.nan,
            "result": res,
            "model": model,
        }
        all_results.append(info)

        if res.success and (best is None or -res.fun > best["q_min"]):
            best = info

    return best, all_results

## 3.7 Reporting Helpers

These functions compute the achieved qualities and display the non-zero decision variables so the final answer can be reported clearly.

In [12]:
def solution_to_dataframe(x, tol=1e-9):
    names = (
        list(barley["name"])
        + list(malts["name"])
        + list(maltextracts["name"])
        + list(hops["name"])
        + list(yeasts["name"])
        + ["q_min", "z_malt", "z_mash"]
    )
    df = pd.DataFrame({"variable": names, "value": x})
    return df[df["value"].abs() > tol].reset_index(drop=True)

def summarize_solution(model, res):
    x = res.x
    b = x[slice_B()]
    m = x[slice_M()]
    e = x[slice_E()]
    h = x[slice_H()]
    y = x[slice_Y()]

    m_tot = model["m_tot"]
    H_tot = model["H_tot"]
    Y_tot = model["Y_tot"]
    gamma = (SG - 1.0) / 0.0344

    ebc_value = gamma * (
        BARLEY_TO_MALTEXTRACT_EQ * np.sum(barley_ebc * b)
        + MALT_TO_MALTEXTRACT_EQ * np.sum(malt_ebc * m)
        + np.sum(extract_ebc * e)
    ) / m_tot if m_tot > 0 else 0.0

    q_malt_value = (
        BARLEY_TO_MALTEXTRACT_EQ * np.sum(barley_q * b)
        + MALT_TO_MALTEXTRACT_EQ * np.sum(malt_q * m)
        + np.sum(extract_q * e)
    ) / m_tot if m_tot > 0 else 0.0
    q_hop_value = np.sum(hop_q * h) / H_tot if H_tot > 0 else 0.0
    q_yeast_value = np.sum(yeast_q * y) / Y_tot if Y_tot > 0 else 0.0

    return {
        "q_min": x[iQ],
        "EBC": ebc_value,
        "Q_malt": q_malt_value,
        "Q_hop": q_hop_value,
        "Q_yeast": q_yeast_value,
        "z_malt": x[iZm],
        "z_mash": x[iZs],
    }

def print_solution(title, model, res):
    print(f"\n=== {title} ===")
    print("Success:", res.success)
    print("Message:", res.message)
    if not res.success:
        return
    summary = summarize_solution(model, res)
    for k, v in summary.items():
        print(f"{k}: {v}")
    print("\nNon-zero variables:")
    print(solution_to_dataframe(res.x).to_string(index=False))

## 3.7 Run the Model

This final block solves the quality-maximization problem for 5000 litres of amber beer and shows the best process choice together with the resulting qualities.

In [13]:
V_beer = 5000
EBC_min = 20
EBC_max = 29

In [14]:
best_solution, all_results = solve_bruteforce(V_beer, EBC_min, EBC_max)

print("Brute-force cases:")
print(pd.DataFrame([
    {"z_fix": r["z_fix"], "success": r["success"], "q_min": r["q_min"]}
    for r in all_results
]).to_string(index=False))

if best_solution is not None:
    print_solution("3.7 Maximum minimum quality solution", best_solution["model"], best_solution["result"])

Brute-force cases:
 z_fix  success    q_min
(0, 0)     True 3.234165
(0, 1)     True 5.000000
(1, 1)     True 5.000000

=== 3.7 Maximum minimum quality solution ===
Success: True
Message: Optimization terminated successfully. (HiGHS Status 7: Optimal)
q_min: 5.0
EBC: 20.000000000000004
Q_malt: 4.999999999999998
Q_hop: 5.0
Q_yeast: 5.0
z_malt: 0.0
z_mash: 1.0

Non-zero variables:
     variable      value
       malt11 870.771733
maltextract03  66.540509
        hop10   6.842105
      yeast01   3.750000
        q_min   5.000000
       z_mash   1.000000


<!-- REPORT_TABLE_OUTPUTS_SYNC -->
## Report table outputs

The following tables correspond to the tables used in the LaTeX report for Task 3.7.

In [15]:
# REPORT_TABLE_OUTPUTS_SYNC
summary = summarize_solution(best_solution["model"], best_solution["result"])
print("Table 10.1: Task 7 Quality Maximisation Result")
task37_quality_table = pd.DataFrame([
    {"Item": "Maximum minimum quality", "Result": f"{summary['q_min']:.3f}"},
    {"Item": "Quality 5 feasible?", "Result": "yes" if summary["q_min"] >= 5 - 1e-8 else "no"},
    {"Item": "Process choice", "Result": f"({int(round(summary['z_malt']))},{int(round(summary['z_mash']))})"},
    {"Item": "Achieved EBC", "Result": f"{summary['EBC']:.3f}"},
    {"Item": "Malt-source quality", "Result": f"{summary['Q_malt']:.3f}"},
    {"Item": "Hop quality", "Result": f"{summary['Q_hop']:.3f}"},
    {"Item": "Yeast quality", "Result": f"{summary['Q_yeast']:.3f}"},
])
display(task37_quality_table)

def report_nonzero_ingredients(x, tol=1e-7):
    groups = [
        ("Barley", list(barley["name"]), x[slice_B()]),
        ("Malt", list(malts["name"]), x[slice_M()]),
        ("Malt extract", list(maltextracts["name"]), x[slice_E()]),
        ("Hops", list(hops["name"]), x[slice_H()]),
        ("Yeast", list(yeasts["name"]), x[slice_Y()]),
    ]
    rows = []
    for group, names, values in groups:
        for name, value in zip(names, values):
            if abs(value) > tol:
                rows.append({"Group": group, "Variable": name, "Amount (kg)": value})
    return pd.DataFrame(rows).round({"Amount (kg)": 3})

print("Table 10.2: Task 7 Non-Zero Ingredients")
task37_ingredients_table = report_nonzero_ingredients(best_solution["result"].x)
display(task37_ingredients_table)


Table 10.1: Task 7 Quality Maximisation Result


,Item,Result
0,Maximum minimum quality,5.000
1,Quality 5 feasible?,yes
2,Process choice,"(0,1)"
3,Achieved EBC,20.000
4,Malt-source quality,5.000
5,Hop quality,5.000
6,Yeast quality,5.000


Table 10.2: Task 7 Non-Zero Ingredients


,Group,Variable,Amount (kg)
0,Malt,malt11,870.772
1,Malt extract,maltextract03,66.541
2,Hops,hop10,6.842
3,Yeast,yeast01,3.750
